# 02 Collect Sources

USGS active streamgages, reviewed streamgage network artifacts, STREAM-geo/NLDI river geometry, direct AORC SST rainfall members, and NWM soil-moisture context.

## Stage Contract
Requires: Location config, source-collection credentials/cache as available, and region setup artifacts.
Produces: reviewed source inventories, forcing-source artifacts, and collection QA tables for the Greensboro inland workflow.
Next: Build the design event catalog after source readiness is clean.


In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

from study_location import bootstrap
session = bootstrap()
location_root = session.location_root
repo_root = session.repo_root

# plan reruns, reuse reviewed data, and audit readiness.
import collect_sources.workflow as collect
from collect_sources.nwm import soil_moisture_csv_has_variables
from collect_sources.usgs_streamgages import discover_active_streamgage_candidates, write_reviewed_streamgage_network

runtime = collect.load_runtime(location_root)
config = runtime.runtime_config
paths = runtime.runtime_paths
reviewed_network_path = runtime.reviewed_network_path

display(collect.summary(config, paths))
display(collect.source_records(runtime))


### Rerun


In [ ]:
rerun = False
source_skip_existing = not rerun


## Source Collection Plan

In [ ]:
plan = collect.plan(config, paths)
plan_table = collect.plan_table(
    plan,
    paths,
    rerun=rerun,
)
display(plan_table)

## USGS Active Streamgage Discovery

In [ ]:
discovery_summary, records_summary = collect.discover_gages(runtime)
display(discovery_summary)
display(records_summary)
display(collect.gage_readiness(runtime))


## Stochastic Storm Transposition Region

The SST region is defined in `config.yaml`

In [ ]:
# Plot the configured AORC SST transposition region before pulling rainfall members.
fig, ax = collect.plot_sst_region(config, paths)

## STREAM-geo/NLDI River Geometry

STREAM-geo width/depth estimates are cached for native Wflow river-geometry enrichment; NLDI is retained as the COMID lookup provenance path.


In [ ]:
display(collect.stream_sources(config, paths))


### Reservoir Condition Inputs -- location dependent

Show configured reservoir-condition sources when an inland watershed includes regulated storage; otherwise this records that reservoir collection is disabled.


In [ ]:
display(collect.reservoir_sources(config, paths))


## Direct AORC SST Rainfall Members

The direct AORC SST collector scans the transposition region, ranks rolling storm windows, and writes the rainfall-member table for precipitation pairing with streamflow and antecedent soil-moisture states.


In [ ]:
# --- AORC SST collection parameters (edit to retune, then run Run Collection below) ---
# Threshold-driven POT: keep every INDEPENDENT storm whose 72h footprint-mean depth
# exceeds the threshold, so the rainfall-member count is data-driven
# Set the threshold from the rainfall POT diagnostics; raise it for fewer, heavier members.
min_precip_threshold = 50.0   # mm over the 72h storm window (footprint mean)
decluster_hours = 72          # minimum spacing between independent storms
storm_duration_hours = 72     # SST rainfall accumulation window
check_every_n_hours = 6       # transposition scan stride
defer_event_windows = True    # write AORC event-window NetCDFs, including Wflow meteo, in the separate cell below

display(collect.aorc_sst_params(
    config,
    paths,
    min_precip_threshold=min_precip_threshold,
    decluster_hours=decluster_hours,
    storm_duration_hours=storm_duration_hours,
    check_every_n_hours=check_every_n_hours,
    defer_event_windows=defer_event_windows,
))


## NWM Soil-Moisture Context

Selected NWM LDAS soil-moisture cells are retained for antecedent-condition pairing. Inland streamflow frequency uses reviewed USGS gages, so NWM streamflow remains context rather than the production frequency source.


In [ ]:
soil_variables = config["collection"]["nwm"]["soil_moisture"]["variables"]
soil_moisture_path = runtime.resolve_location_path(config["event_catalog"]["forcing_members"]["soil_moisture"])
soil_moisture_ready = soil_moisture_csv_has_variables(soil_moisture_path, soil_variables)
display(collect.soil_sources(config, paths))
display(pd.Series({"soil_moisture_path": soil_moisture_path, "soil_moisture_ready": soil_moisture_ready}, name="nwm_soil_moisture_readiness"))

## Run Collection

In [ ]:
# Collect any missing configured sources and summarize the artifacts.
collectable_readiness = collect.readiness(runtime)
collect_missing_sources = rerun or not collectable_readiness["ready"].all()
prerequisite_result = collect.prepare(config, paths)
collection_result = collect.run_collect(
    config,
    paths,
    plan,
    run_collection=collect_missing_sources,
    skip_existing=source_skip_existing,
    stop_on_error=False,
    progress=True,
)
display(pd.concat([
    collectable_readiness.assign(table="pre_collection_readiness"),
    prerequisite_result.assign(table="collection_prerequisite"),
    collection_result.assign(table="collection_result"),
], ignore_index=True, sort=False))


### AORC SST Event Windows

Materialize per-storm AORC event-window NetCDFs with paired precipitation and Wflow meteo after the rainfall member table is current.


In [ ]:
# Materialize per-storm AORC event-window NetCDFs with paired precip + Wflow meteo.
aorc_event_window_result = collect.collect_aorc_sst_event_windows(
    config,
    paths,
    plan,
    skip_existing=source_skip_existing,
)
display(aorc_event_window_result)

aorc_event_window_readiness = collect.aorc_sst_event_window_readiness(config, paths, plan)
display(aorc_event_window_readiness)
if aorc_event_window_readiness["status"] != "ready":
    raise RuntimeError(
        "AORC event windows are not ready for paired Wflow/SFINCS forcing: "
        f"{aorc_event_window_readiness.to_dict()}"
    )


### Reservoir Collection Review


In [ ]:
reservoir_conditions = collect.reservoir_condition_table(config, paths)
display(reservoir_conditions)


## Reviewed Streamgage Network Writer

In [ ]:
write_reviewed_streamgage_network_file = True
reviewed_network_decisions = collect.build_reviewed_streamgage_decisions(config, paths)
reviewed_streamgage_decision_table = pd.DataFrame(reviewed_network_decisions)
if write_reviewed_streamgage_network_file:
    reviewed_network_result = write_reviewed_streamgage_network(config, paths, reviewed_network_decisions)
else:
    reviewed_network_result = {
        "status": "review_pending",
        "reviewed_network_geojson": str(runtime.reviewed_network_path),
        "accepted_count": int(reviewed_streamgage_decision_table["review_status"].str.startswith("accepted").sum()),
        "reason": "Review candidate gages, then set write_reviewed_streamgage_network_file=True.",
    }

fig, artifact_summary, gage_domain_summary = collect.plot_usgs_streamgage_network(runtime)
reviewed_records_result = collect.collect_gage_records(runtime, skip_existing=source_skip_existing)

display(reviewed_streamgage_decision_table)
display(pd.Series(reviewed_network_result, name="reviewed_streamgage_network"))
display(pd.Series(reviewed_records_result, name="usgs_reviewed_discharge_records"))
display(artifact_summary)
display(gage_domain_summary)


## Collected Data Overview

In [ ]:
display(collect.overview())

### SST

Storm transposition targets are plotted against the configured SST region and study area.


In [ ]:
# Plot the SST region, study area, and rainfall transposition targets.
fig, ax = collect.plot_collected_sst_geography(config, paths)

### NWM Soil Moisture

Soil moisture is summarized across configured NWM points and layers.


In [ ]:
# Plot monthly mean NWM soil moisture variables when available.
fig, status = collect.plot_nwm_soil_moisture(config, paths)
display(status)

### AORC SST Rainfall

Rainfall members are shown as selected-event magnitude and distribution summaries.


In [ ]:
# Plot compact AORC SST rainfall member summaries.
fig = collect.plot_aorc_sst_rainfall(paths)